In [ ]:
! pip install langchain langchain-chroma chromadb sentence-transformers transformers accelerate streamlit langchain-huggingface

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma
from transformers import pipeline
import re

In [ ]:
transcripts = {
    "team_sync_2026-03-20": """
    [2026-03-20] John: We should launch the product next Friday.
    [2026-03-20] Sarah: I'll prepare the marketing plan.
    """,
    "team_sync_2026-03-21": """
    [2026-03-21] Mike: The API is still unstable.
    [2026-03-21] John: Let's fix the API before launch.
    """ ,
    "team_sync_2026-03-29": """
    [2026-03-29] Lisa: We decided buy the product on next Monday.
    [2026-03-29] John: Ask the adminstartor to fix Mike's PC.
    """ ,
    "team_sync_2026-03-30": """
    [2026-03-30] Rachel: The product for X company didn't get shipped yesterday.
    [2026-03-30] John: We should do meeting once every week.
    """
}

In [ ]:
docs = []

def extract_date(text):
  match = re.search(r"\[(.*?)]" , text)
  return match.group(1) if match else None

seen = set()

for meeting_id , text in transcripts.items():
  chunks = [line.strip() for line in text.split("\n") if line.strip()]



  for chunk in chunks:
    if chunk not in seen:
      seen.add(chunk)
      docs.append(Document(
          page_content = chunk ,
          metadata ={

              "date" : extract_date(chunk) ,
              "meeting_id" : meeting_id
        }
  ))


for d in docs:
  print(d.page_content , d.metadata)




[2026-03-20] John: We should launch the product next Friday. {'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-20] Sarah: I'll prepare the marketing plan. {'date': '2026-03-20', 'meeting_id': 'team_sync_2026-03-20'}
[2026-03-21] Mike: The API is still unstable. {'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
[2026-03-21] John: Let's fix the API before launch. {'date': '2026-03-21', 'meeting_id': 'team_sync_2026-03-21'}
[2026-03-29] Lisa: We decided buy the product on next Monday. {'date': '2026-03-29', 'meeting_id': 'team_sync_2026-03-29'}
[2026-03-29] John: Ask the adminstartor to fix Mike's PC. {'date': '2026-03-29', 'meeting_id': 'team_sync_2026-03-29'}
[2026-03-30] Rachel: The product for X company didn't get shipped yesterday. {'date': '2026-03-30', 'meeting_id': 'team_sync_2026-03-30'}
[2026-03-30] John: We should do meeting once every week. {'date': '2026-03-30', 'meeting_id': 'team_sync_2026-03-30'}


In [ ]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = Chroma.from_documents(docs , embedding)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import re
from datetime import datetime, timedelta

def parse_query(query):
    query_lower = query.lower()
    time_filter = None
    today = datetime.today().date()

    range_match = re.search(r"(\d{4}-\d{2}-\d{2}).*?(\d{4}-\d{2}-\d{2})", query_lower)
    if range_match:
        return {"type": "date_range", "start": range_match.group(1), "end": range_match.group(2)}

    exact_match = re.search(r"\d{4}-\d{2}-\d{2}", query_lower)
    if exact_match:
        return {"type": "exact_date", "date": exact_match.group(0)}

    natural_match = re.search(
        r"(\d{1,2})\s+(january|february|march|april|may|june|july|august|september|october|november|december)(?:\s+(\d{4}))?",
        query_lower
    ) or re.search(
        r"(january|february|march|april|may|june|july|august|september|october|november|december)\s+(\d{1,2})(?:\s+(\d{4}))?",
        query_lower
    )

    if natural_match:
        groups = natural_match.groups()
        try:
            if groups[0].isdigit():
                day, month_str, year = groups[0], groups[1], groups[2] or str(today.year)
            else:
                month_str, day, year = groups[0], groups[1], groups[2] or str(today.year)
            parsed = datetime.strptime(f"{day} {month_str} {year}", "%d %B %Y").date()
            return {"type": "exact_date", "date": str(parsed)}
        except ValueError:
            pass

    if "today" in query_lower:
        return {"type": "today", "date": str(today)}

    if "yesterday" in query_lower:
        return {"type": "yesterday", "date": str(today - timedelta(days=1))}

    if "last week" in query_lower:
        return {"type": "last_week", "start": str(today - timedelta(days=7)), "end": str(today)}

    return time_filter

In [ ]:
from datetime import datetime, timedelta

def filter_docs(docs, time_filter=None):
    filtered = docs





    if time_filter:


        if time_filter["type"] == "last_week":
            start = datetime.strptime(time_filter["start"], "%Y-%m-%d")
            end = datetime.strptime(time_filter["end"], "%Y-%m-%d")

            filtered = [
                d for d in filtered
                if d.metadata.get("date")
                and start <= datetime.strptime(d.metadata["date"], "%Y-%m-%d") <= end
            ]


        elif time_filter["type"] == "exact_date":
            target = time_filter["date"]

            filtered = [
                d for d in filtered
                if d.metadata.get("date") == target
            ]

        elif time_filter["type"] in ["today", "yesterday"]:
          target = time_filter["date"]

          filtered = [
              d for d in filtered
              if d.metadata.get("date") == target
            ]


        elif time_filter["type"] == "date_range":
            start = datetime.strptime(time_filter["start"], "%Y-%m-%d")
            end = datetime.strptime(time_filter["end"], "%Y-%m-%d")

            filtered = [
                d for d in filtered
                if d.metadata.get("date")
                and start <= datetime.strptime(d.metadata["date"], "%Y-%m-%d") <= end
            ]

    return filtered

In [ ]:
from chromadb import EphemeralClient

def hybrid_search(query, db, filtered_docs, k=3):
    if not filtered_docs:
        return []

    client = EphemeralClient()
    temp_db = Chroma.from_documents(
        filtered_docs,
        embedding,
        client=client,
        collection_name="temp_search"
    )

    semantic_docs = temp_db.similarity_search(query, k=k)

    seen = set()
    combined = []
    for d in semantic_docs:
        if d.page_content not in seen:
            combined.append(d)
            seen.add(d.page_content)

    return combined[:k]

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-xl"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

def rag_pipeline(query):
    time_filter = parse_query(query)
    filtered_docs = filter_docs(docs, time_filter)

    if not filtered_docs:
        return "I don't know", []

    results = hybrid_search(query, db, filtered_docs)

    if not results:
        return "I don't know", []

    seen = set()
    unique_results = []
    for r in results:
        if r.page_content not in seen:
            unique_results.append(r)
            seen.add(r.page_content)

    results = unique_results
    context = "\n".join([r.page_content for r in results])
    query_clean = query.strip()

    prompt = f"""You are a meeting assistant.
Use the context to answer the question.
If no answer exists, reply exactly: "No records found."

Context:
{context}

Question:
{query_clean}

Summarize the key points as a short list:"""



    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer, results

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
query = "actions taken on 21 march by john "

answer , results = rag_pipeline(query)

print("ANSWER :\n", answer)
print("\nRETRIEVED CHUNKS:\n")

for r in results:
  print(r.page_content)

ANSWER :
 Fix the API before launch

RETRIEVED CHUNKS:

[2026-03-20] John: We should launch the product next Friday.
[2026-03-21] John: Let's fix the API before launch.
